# Manifest Package Operator Tutorial

This notebook teaches the complete workflow for preparing video metadata and later publishing final manifests to ADLS Gen2.

The workflow is intentionally split:

1. **Preparation** discovers and validates videos, calculates hashes, and writes a destination-independent manifest package. It requires no Azure access.
2. **Publication** verifies the package and the exact video bytes, uploads each video, obtains its final URI and ETag, and publishes the final manifest last.

> **Safety:** publication is disabled by default. No cell contacts Azure unless `RUN_PUBLICATION` is explicitly changed to `True`.

## 1. Roles and prerequisites

The same person may perform both roles at different times, or the package may be transferred to another operator.

| Role | Needs source videos | Needs camera catalog | Needs Azure write access |
|---|---:|---:|---:|
| Preparation operator | Yes | Yes | No |
| Publication operator | Yes, unchanged bytes | No | Yes |

Host prerequisites:

- Python 3.10 or newer and `uv`;
- this repository and its installed environment;
- FFmpeg's `ffprobe` on `PATH`;
- for publication only, the `publisher` extra and an Azure identity with narrowly scoped `Storage Blob Data Contributor` access.

Never place credentials in the catalog, exception inventory, package, notebook, command history, or logs.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import subprocess


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "samples").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the people-counter repository")


REPO_ROOT = find_repo_root()
VIDEO_ROOT = REPO_ROOT
PARTITION_PREFIX = "samples/"
INPUT_DIR = REPO_ROOT / "samples" / "manifest-package-tutorial-inputs"
PACKAGE_DIR = REPO_ROOT / "samples" / "manifest-package-tutorial"
CATALOG_PATH = INPUT_DIR / "camera_catalog.csv"
EXCEPTION_INVENTORY_PATH = INPUT_DIR / "video-inventory-exceptions.csv"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
assert PACKAGE_DIR.parent == REPO_ROOT / "samples"

print(f"Repository: {REPO_ROOT}")
print(f"Video root: {VIDEO_ROOT}")
print(f"Package output: {PACKAGE_DIR}")

## 2. Create the camera metadata catalog

Create one effective-dated row for each camera configuration, not one row per video. A row defines:

- the source path prefix used to associate videos with a camera;
- stable camera and location identifiers;
- the IANA camera timezone;
- expected frame dimensions;
- the directed counting line;
- how capture time is resolved; and
- the effective UTC interval for that configuration.

Use `capture_time_source=auto` in most cases. Resolution order is:

1. an optional exception-inventory override;
2. embedded stream/container `creation_time`;
3. the configured filename regular expression.

The counting line direction matters: reversing its endpoints reverses `in` and `out`. Coordinates must fit within the declared frame dimensions.

In [ ]:
catalog_fields = [
    "catalog_version", "source_path_prefix", "camera_id", "location_id",
    "camera_timezone", "frame_width", "frame_height",
    "counting_line_x1", "counting_line_y1", "counting_line_x2", "counting_line_y2",
    "capture_time_source", "capture_time_regex", "capture_time_format",
    "effective_from_utc", "effective_to_utc",
]

# The sample clips have different dimensions and effective capture dates.
catalog_rows = [
    {
        "catalog_version": 1, "source_path_prefix": "samples/",
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 3626, "frame_height": 1994,
        "counting_line_x1": 0, "counting_line_y1": 997,
        "counting_line_x2": 3625, "counting_line_y2": 997,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-01-01T00:00:00Z",
        "effective_to_utc": "2023-09-28T00:00:00Z",
    },
    {
        "catalog_version": 1, "source_path_prefix": "samples/",
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 3840, "frame_height": 2160,
        "counting_line_x1": 0, "counting_line_y1": 1080,
        "counting_line_x2": 3839, "counting_line_y2": 1080,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-09-28T00:00:00Z",
        "effective_to_utc": "2023-10-01T00:00:00Z",
    },
    {
        "catalog_version": 1, "source_path_prefix": "samples/",
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 1920, "frame_height": 1080,
        "counting_line_x1": 0, "counting_line_y1": 540,
        "counting_line_x2": 1919, "counting_line_y2": 540,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-10-01T00:00:00Z",
        "effective_to_utc": "",
    },
]

with CATALOG_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=catalog_fields)
    writer.writeheader()
    writer.writerows(catalog_rows)

print(CATALOG_PATH.read_text(encoding="utf-8"))

## 3. Preview source-video metadata

Preparation performs this inspection automatically. The preview below helps an operator understand which timestamps can be derived without manual input.

Do not use filesystem modification time or upload time as recording time. If embedded metadata and filename rules cannot provide a trustworthy timestamp, preparation rejects that video with an actionable reason code.

In [ ]:
video_paths = sorted((REPO_ROOT / "samples").glob("*.mp4"))
if not video_paths:
    raise RuntimeError("No sample MP4 videos were found")

for video_path in video_paths:
    completed = subprocess.run(
        [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries",
            "stream=width,height,duration:stream_tags=creation_time:format=duration:format_tags=creation_time",
            "-of", "json", str(video_path),
        ],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    metadata = json.loads(completed.stdout)
    stream = metadata["streams"][0]
    embedded_time = stream.get("tags", {}).get("creation_time")
    duration = stream.get("duration") or metadata.get("format", {}).get("duration")
    print(
        video_path.relative_to(REPO_ROOT),
        f"{stream['width']}x{stream['height']}",
        f"duration={duration}",
        f"creation_time={embedded_time}",
    )

## 4. Run preparation without exception overrides

The first pass intentionally omits `--inventory`. Two sample videos have embedded timestamps; one does not. Exit code `2` means preparation completed but at least one video was rejected.

An incomplete package is never publishable. Review grouped totals in `summary.json` and individual remediation in `rejection-report.csv`.

In [ ]:
prepare_base = [
    "uv", "run", "prepare-manifests",
    "--catalog", str(CATALOG_PATH),
    "--video-root", str(VIDEO_ROOT),
    "--partition-prefix", PARTITION_PREFIX,
    "--output-dir", str(PACKAGE_DIR),
    "--max-files", "10",
]

first_pass = subprocess.run(
    prepare_base,
    cwd=REPO_ROOT,
    check=False,
    capture_output=True,
    text=True,
)
print(first_pass.stdout.strip())
if first_pass.stderr.strip():
    print(first_pass.stderr.strip())
if first_pass.returncode not in {0, 2}:
    raise RuntimeError(f"Preparation failed with exit code {first_pass.returncode}")
print(f"Exit code: {first_pass.returncode}")

In [ ]:
summary = json.loads((PACKAGE_DIR / "summary.json").read_text(encoding="utf-8"))
with (PACKAGE_DIR / "rejection-report.csv").open(newline="", encoding="utf-8") as handle:
    rejections = list(csv.DictReader(handle))

print(json.dumps(summary, indent=2, sort_keys=True))
print("\nRejections:")
for rejection in rejections:
    print(
        f"- {rejection['video_relative_path']}: {rejection['reason_code']}\n"
        f"  Explanation: {rejection['explanation']}\n"
        f"  Suggested action: {rejection['suggested_action']}"
    )

## 5. Add only exceptional capture-time overrides

Do not manually inventory all videos. The optional inventory exists only for files whose trustworthy capture time cannot be derived by code.

The timestamp below is a tutorial value. In production, obtain it from an authoritative camera or source-system record and include an explicit UTC `Z` or numeric offset.

In [ ]:
exception_rows = [
    {
        "video_relative_path": "samples/three_people_walking.mp4",
        "captured_at_utc": "2023-10-01T01:00:00Z",
    }
]

with EXCEPTION_INVENTORY_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["video_relative_path", "captured_at_utc"],
    )
    writer.writeheader()
    writer.writerows(exception_rows)

print(EXCEPTION_INVENTORY_PATH.read_text(encoding="utf-8"))

In [ ]:
complete_pass = subprocess.run(
    [
        *prepare_base,
        "--inventory", str(EXCEPTION_INVENTORY_PATH),
    ],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(complete_pass.stdout.strip())

## 6. Inspect the completed manifest package

`manifest-package.json` is the authoritative checksum index. Prepared manifests intentionally omit `video_uri` and `source_etag`; publication adds those storage-dependent values later.

Identity is deterministic:

```text
asset_id      = SHA256(normalized source-relative path)
asset_version = SHA256(video bytes)
```

Changing one byte creates a new asset version. Correcting bytes at the same relative path retains the logical asset ID.

In [ ]:
package_index = json.loads(
    (PACKAGE_DIR / "manifest-package.json").read_text(encoding="utf-8")
)
package_summary = json.loads(
    (PACKAGE_DIR / "summary.json").read_text(encoding="utf-8")
)
with (PACKAGE_DIR / "generated-video-inventory.csv").open(
    newline="", encoding="utf-8"
) as handle:
    generated_inventory = list(csv.DictReader(handle))

print("Package files:")
for path in sorted(PACKAGE_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(PACKAGE_DIR))

print("\nIndex summary:")
print(json.dumps({
    key: package_index[key]
    for key in ("manifest_package_version", "complete", "discovered", "prepared", "rejected")
}, indent=2))
assert package_index["complete"] is True
assert package_index["prepared"] == len(generated_inventory) == 3
assert not list(PACKAGE_DIR.rglob("*.mp4"))

sample_entry = package_index["entries"][0]
sample_prepared_path = PACKAGE_DIR / sample_entry["prepared_manifest_path"]
sample_prepared = json.loads(sample_prepared_path.read_text(encoding="utf-8"))
assert "video_uri" not in sample_prepared
assert "source_etag" not in sample_prepared
print("\nExample prepared manifest:")
print(json.dumps(sample_prepared, indent=2, sort_keys=True))

In [ ]:
for row in generated_inventory:
    relative_path = row["video_relative_path"]
    video_path = VIDEO_ROOT.joinpath(*relative_path.split("/"))
    expected_asset_id = hashlib.sha256(relative_path.encode("utf-8")).hexdigest()
    video_digest = hashlib.sha256()
    with video_path.open("rb") as source:
        while chunk := source.read(8 * 1024 * 1024):
            video_digest.update(chunk)
    assert row["asset_id"] == expected_asset_id
    assert row["asset_version"] == video_digest.hexdigest()
    assert row["sha256"] == row["asset_version"]
    print(relative_path, row["asset_version"])

print("All generated identities match the source paths and exact video bytes.")

## 7. Validate exactly as publication will

This validation is offline: it reads the package and rehashes every referenced video before any Azure client is created. It catches edited package files, missing videos, changed bytes, unsupported versions, duplicate identities, inconsistent counts, and incomplete preparation.

In [ ]:
from people_counter.manifest_publisher import (
    ManifestPackagePublishConfig,
    validate_manifest_package,
)

validation_config = ManifestPackagePublishConfig(
    manifest_package_dir=PACKAGE_DIR,
    video_root=VIDEO_ROOT,
    storage_account="validation-only",
    filesystem="validation-only",
    staging_prefix="staging",
    incoming_prefix="incoming",
    checkpoint_path=PACKAGE_DIR / "unused-publication.sqlite3",
    rejection_report_path=PACKAGE_DIR / "unused-publication-rejections.csv",
    summary_report_path=PACKAGE_DIR / "unused-publication-summary.json",
    chunk_size=8 * 1024 * 1024,
)

validated_package = validate_manifest_package(validation_config)
print(f"Validated {len(validated_package.plans)} videos without Azure access.")

## 8. Retain or transfer the package

Publication may happen immediately or later. The publication operator needs:

1. the unchanged source-video directory tree; and
2. the complete manifest package directory.

The absolute path may change, but relative paths under `--video-root` must remain the same. SHA-256 integrity is authoritative: publication rehashes the video files and refuses changed or missing bytes.

Do not edit prepared JSON files or `manifest-package.json`. Correct source/catalog data and rerun preparation instead.

## 9. Configure publication access

The publication host needs the optional dependencies:

```bash
uv sync --extra publisher
```

`publish-manifests` uses Azure `DefaultAzureCredential`. Supported credential sources include managed identity, workload identity, an attended Azure CLI login, or environment-provided service-principal variables. Never put secrets in this notebook.

Grant the publisher identity `Storage Blob Data Contributor` on the target filesystem, or equivalent narrowly scoped ADLS permissions for `staging/` and `incoming/`.

Publication validates the entire package and local video hashes before creating the Azure client. It uploads and renames each video first, then creates and renames its final manifest last.

In [ ]:
RUN_PUBLICATION = False
STORAGE_ACCOUNT = "replace-with-storage-account"
FILESYSTEM = "replace-with-filesystem"

publish_command = [
    "uv", "run", "--extra", "publisher", "publish-manifests",
    "--manifest-package-dir", str(PACKAGE_DIR),
    "--video-root", str(VIDEO_ROOT),
    "--storage-account", STORAGE_ACCOUNT,
    "--filesystem", FILESYSTEM,
    "--staging-prefix", "staging",
    "--incoming-prefix", "incoming",
    "--checkpoint", str(PACKAGE_DIR / "publication.sqlite3"),
    "--rejection-report", str(PACKAGE_DIR / "publication-rejections.csv"),
    "--summary-report", str(PACKAGE_DIR / "publication-summary.json"),
]

print("Publication command:")
print(" ".join(publish_command))

if RUN_PUBLICATION:
    if STORAGE_ACCOUNT.startswith("replace-") or FILESYSTEM.startswith("replace-"):
        raise ValueError("Set STORAGE_ACCOUNT and FILESYSTEM before enabling publication")
    subprocess.run(publish_command, cwd=REPO_ROOT, check=True)
else:
    print("Publication is disabled. Set RUN_PUBLICATION=True only after reviewing all settings.")

## 10. Verify publication and register backfill work

After publication:

1. Require publication exit code `0` and `rejected=0` in `publication-summary.json`.
2. Verify each final video and JSON manifest exists under the expected `incoming/<date>/<asset-id>/<asset-version>/` path.
3. Confirm manifests contain the final ADLS `video_uri`, the final video ETag, expected size, and expected SHA-256.
4. Do not mutate objects under `incoming/`; corrected bytes create another content-hash asset version.
5. Run the Fabric `pc-backfill-register` pipeline once per bounded manifest partition, as documented in `notebooks/fabric/README.md`.
6. Reconcile source videos, package entries, published manifests, registered work, committed work, and dead-lettered work before declaring the backfill complete.

## 11. Troubleshooting guide

| Reason code | Action |
|---|---|
| `CAPTURE_TIME_MISSING` | Repair embedded metadata, add a filename timestamp rule, or add only affected videos to the exception inventory |
| `CAPTURE_TIME_FILENAME_MISMATCH` | Correct the catalog regex/format or supply an authoritative override |
| `NO_CAMERA_MATCH` | Add or correct the catalog `source_path_prefix` |
| `CAMERA_EFFECTIVE_RANGE_MISMATCH` | Correct the timestamp or effective-dated catalog rows |
| `VIDEO_DIMENSION_MISMATCH` | Correct catalog dimensions/counting line or add another effective-dated configuration |
| `VIDEO_PROBE_FAILED` | Validate, repair, or remux the source video |
| `MANIFEST_PACKAGE_INCOMPLETE` | Resolve all preparation rejections and regenerate the package |
| `MANIFEST_PACKAGE_CHECKSUM_MISMATCH` | Recopy or regenerate the package; never edit prepared files |
| `MANIFEST_PACKAGE_VIDEO_MISMATCH` | Restore the exact source bytes or regenerate from the current video tree |
| `PUBLICATION_CONFLICT` | Investigate the existing ADLS object/checkpoint; never overwrite `incoming/` |

For large backfills, use the grouped rejection counts in `summary.json` to fix the largest issue classes first instead of reviewing console logs one by one.